In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import duckdb


In [ ]:
import duckdb
import glob
import os

# Load the Parquet file using DuckDB
con = duckdb.connect()

print("=" * 80)
print("EXTREME DEBUG MODE")
print("=" * 80)

# 1. Where are we RIGHT NOW?
print("\n1. CURRENT LOCATION:")
print(f"   os.getcwd(): {os.getcwd()}")
print(f"   os.listdir('.'): {os.listdir('.')[:10]}")  # First 10 items

# 2. Can we see the 1_LIB folder from here?
print("\n2. CHECKING FOR 1_LIB:")
for item in os.listdir('.'):
    print(f"   - {item} {'(dir)' if os.path.isdir(item) else '(file)'}")

# 3. Walk the entire tree looking for ANY .parquet file
print("\n3. SEARCHING ENTIRE TREE FOR .parquet FILES:")
found_parquets = []
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.parquet'):
            full_path = os.path.join(root, file)
            found_parquets.append(full_path)
            print(f"   FOUND: {full_path}")

if not found_parquets:
    print("   NO .parquet FILES FOUND ANYWHERE!")
else:
    print(f"\n   Total parquet files found: {len(found_parquets)}")

# 4. If we found any, show their common ancestor
if found_parquets:
    print("\n4. PARQUET FILE LOCATIONS:")
    for p in found_parquets[:5]:  # Show first 5
        print(f"   {os.path.abspath(p)}")
    
    # Extract the pattern
    first_parquet = found_parquets[0]
    print(f"\n5. FIRST PARQUET FILE:")
    print(f"   Relative: {first_parquet}")
    print(f"   Absolute: {os.path.abspath(first_parquet)}")
    
    # Now try to load them
    print("\n6. ATTEMPTING TO LOAD WITH DUCKDB:")
    
    try:
        # Just use the list of files we found
        files_list = ", ".join([f"'{os.path.abspath(f)}'" for f in found_parquets])
        
        query = f"""
            SELECT "Time Stamp", Load
            FROM read_parquet([{files_list}])
        """
        
        print(f"   Loading {len(found_parquets)} files...")
        df = con.execute(query).df()
        print(f"   ✓ SUCCESS! Loaded {len(df)} rows")
        print(f"\n   DataFrame shape: {df.shape}")
        print(f"\n   First few rows:")
        print(df.head())
        
    except Exception as e:
        print(f"   ✗ FAILED: {e}")
        print(f"\n   Trying alternative method...")
        
        # Try just the first file to test
        try:
            test_query = f"""
                SELECT "Time Stamp", Load
                FROM read_parquet('{os.path.abspath(found_parquets[0])}')
            """
            df_test = con.execute(test_query).df()
            print(f"   ✓ Single file works! Loaded {len(df_test)} rows from first file")
            
            # Now try all files one by one
            print(f"\n   Loading all {len(found_parquets)} files individually...")
            dfs = []
            for pf in found_parquets:
                try:
                    df_single = con.execute(f"""
                        SELECT "Time Stamp", Load
                        FROM read_parquet('{os.path.abspath(pf)}')
                    """).df()
                    dfs.append(df_single)
                    print(f"   ✓ Loaded {pf} ({len(df_single)} rows)")
                except Exception as e:
                    print(f"   ✗ Failed {pf}: {e}")
            
            # Concatenate all dataframes
            import pandas as pd
            df = pd.concat(dfs, ignore_index=True)
            print(f"\n   ✓ COMBINED SUCCESS! Total rows: {len(df)}")
            
        except Exception as e2:
            print(f"   ✗ Even single file failed: {e2}")

else:
    # No parquet files found at all - show the directory structure
    print("\n4. FULL DIRECTORY STRUCTURE (first 3 levels):")
    for root, dirs, files in os.walk('.'):
        level = root.replace('.', '').count(os.sep)
        if level < 3:
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 2 * (level + 1)
            for file in files[:10]:  # First 10 files per dir
                print(f'{subindent}- {file}')
    
    print("No parquet files found anywhere in the directory tree!")

print("\n" + "=" * 80)

In [ ]:
import duckdb
import glob
import os

# Load the Parquet file using DuckDB
con = duckdb.connect()

# Debug: Print current location and environment
print("=" * 60)
print("DEBUG: Current working directory:", os.getcwd())
print("DEBUG: __file__ location:", os.path.abspath(__file__) if '__file__' in globals() else "Not available (notebook)")
print("DEBUG: GITHUB_WORKSPACE:", os.environ.get('GITHUB_WORKSPACE', 'Not set'))
print("=" * 60)

# Strategy: Try multiple possible paths
possible_paths = []

# 1. From notebook location (going up to repo root)
notebook_dir = os.getcwd()
for levels_up in range(5):  # Try going up 0-4 levels
    repo_candidate = os.path.abspath(os.path.join(notebook_dir, *(['..'] * levels_up)))
    parquet_path = os.path.join(repo_candidate, '1_LIB', 'nyiso', 'nyiso_parquet')
    possible_paths.append(parquet_path)

# 2. From GITHUB_WORKSPACE if in CI
if 'GITHUB_WORKSPACE' in os.environ:
    github_path = os.path.join(os.environ['GITHUB_WORKSPACE'], '1_LIB', 'nyiso', 'nyiso_parquet')
    possible_paths.append(github_path)

# 3. Absolute path from error message (Windows CI path)
possible_paths.append(r'D:\a\CS506_Project\CS506_Project\1_LIB\nyiso\nyiso_parquet')

# 4. Direct relative paths
possible_paths.extend([
    './1_LIB/nyiso/nyiso_parquet',
    '../1_LIB/nyiso/nyiso_parquet',
    '../../1_LIB/nyiso/nyiso_parquet',
    '../../../1_LIB/nyiso/nyiso_parquet',
])

# Try each path
parquet_files = []
successful_path = None

print("\nSearching for parquet files in possible locations:")
for path in possible_paths:
    print(f"\nTrying: {path}")
    print(f"  Exists: {os.path.exists(path)}")
    
    if os.path.exists(path):
        pattern = os.path.join(path, '**', '*.parquet')
        files = glob.glob(pattern, recursive=True)
        print(f"  Found {len(files)} parquet file(s)")
        
        if files:
            parquet_files = files
            successful_path = path
            print(f"  ✓ SUCCESS! Using this path")
            break
    else:
        print(f"  Path does not exist")

# Final check
if not parquet_files:
    print("\n" + "=" * 60)
    print("ERROR: Could not find parquet files!")
    print("\nDirectory tree from current location:")
    for root, dirs, files in os.walk('.'):
        level = root.replace('.', '').count(os.sep)
        if level < 3:
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            if any(f.endswith('.parquet') for f in files):
                subindent = ' ' * 2 * (level + 1)
                for f in files:
                    if f.endswith('.parquet'):
                        print(f'{subindent}└─ {f}')
    print("=" * 60)
    raise FileNotFoundError("No parquet files found in any expected location")

print(f"\n✓ Found {len(parquet_files)} parquet files")
print(f"✓ Using path: {successful_path}")
print("\nFirst few files:")
for f in parquet_files[:3]:
    print(f"  - {f}")

# Now load with DuckDB
try:
    # Method 1: Use Python list (safer for paths with special characters)
    files_list = ", ".join([f"'{f}'" for f in parquet_files])
    
    query = f"""
        SELECT "Time Stamp", Load
        FROM read_parquet([{files_list}])
    """
    
    print(f"\nExecuting query with {len(parquet_files)} file(s)...")
    df = con.execute(query).df()
    print(f"✓ Successfully loaded {len(df)} rows")
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nFirst few rows:")
    print(df.head())
    
except Exception as e:
    print(f"\nMethod 1 failed: {e}")
    print("\nTrying Method 2: DuckDB glob pattern...")
    
    # Method 2: Let DuckDB handle the glob
    glob_pattern = os.path.join(successful_path, '**', '*.parquet').replace('\\', '/')
    
    query = f"""
        SELECT "Time Stamp", Load
        FROM read_parquet('{glob_pattern}', hive_partitioning=false)
    """
    
    print(f"Query: {query}")
    df = con.execute(query).df()
    print(f"✓ Successfully loaded {len(df)} rows")

In [ ]:
#need to double check zone coords
zone_coords = {
    "CAPITL": (42.65, -73.75),
    "CENTRL": (43.05, -76.15),
    "DUNWOD": (41.01, -73.78),
    "GENESE": (43.17, -77.61),
    "HUD VL": (41.70, -73.93),
    "MHK VL": (42.10, -75.91),
    "MILLWD": (41.13, -73.78),
    "N.Y.C.": (40.71, -74.01),
    "NORTH": (44.70, -73.45),
    "WEST": (42.89, -78.87)
}

In [ ]:
df

In [ ]:
df_total_load = df.groupby("Time Stamp", as_index=False)["Load"].sum()
df_total_load.rename(columns={"Time Stamp": "Time"}, inplace=True)


In [ ]:
df_total_load

In [ ]:
# Ensure the 'Time' column is a datetime type and set it as the index
df_total_load['Time'] = pd.to_datetime(df_total_load['Time'])
df_total_load.set_index('Time', inplace=True)

# Average hourly load
df_hourly = df_total_load.resample('D').mean().reset_index()
df_hourly = df_hourly.dropna().reset_index()



In [ ]:
df_hourly

In [ ]:
df_total_load = df_hourly

In [ ]:
# Split the data based on the year
train_data = df_total_load[df_total_load['Time'].dt.year.between(2001, 2021)]['Load']
val_data = df_total_load[df_total_load['Time'].dt.year == 2022]['Load']
test_data = df_total_load[df_total_load['Time'].dt.year.isin([2023, 2024, 2025])]['Load']

# Print the sizes of each split
print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Testing data size: {len(test_data)}")

In [ ]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(pd.DataFrame(train_data))
val_scaled = scaler.transform(pd.DataFrame(val_data))
test_scaled = scaler.transform(pd.DataFrame(test_data))

In [ ]:
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        v = X[i:i + time_steps]
        Xs.append(v)
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 5
X_train, y_train = create_dataset(train_scaled, train_scaled, TIME_STEPS)
X_val, y_val = create_dataset(val_scaled, val_scaled, TIME_STEPS)
X_test, y_test = create_dataset(test_scaled, test_scaled, TIME_STEPS)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

In [ ]:
print("NaNs in X_train:", np.isnan(X_train).sum())
print("NaNs in X_val:", np.isnan(X_val).sum())
print("NaNs in y_train:", np.isnan(y_train).sum())
print("NaNs in y_val:", np.isnan(y_val).sum())

In [ ]:
subset_size = 40000
val_subset = 8749
X_train_sub = X_train[:subset_size]
y_train_sub = y_train[:subset_size]
X_val_sub = X_val[:val_subset]
y_val_sub = y_val[:val_subset]


In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
# best_mae = float('inf')
# best_model = None

# for C in [0.1, 1, 10]:
#     for gamma in ['scale', 0.01, 0.001]:
#         for epsilon in [0.01, 0.1, 0.5, 1.0]:
#             print(f"C={C}, gamma={gamma}, epsilon={epsilon}")
#             model = SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon)
#             model.fit(X_train_sub.reshape(X_train_sub.shape[0], -1), y_train_sub)
#             preds = model.predict(X_val_sub.reshape(X_val_sub.shape[0], -1))
#             mae = mean_absolute_error(y_val_sub, preds)
#             print(f"val MAE={mae:.3f}")

#             if mae < best_mae:
#                 best_mae = mae
#                 best_model = model

# print("Best params found:", best_model.get_params())


Best params found: {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}

Best mae: 0.04994650252799576

In [ ]:
best_params = {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}
best_model = SVR(**best_params)
best_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)

In [ ]:

# Make predictions
train_pred = best_model.predict(X_train.reshape(X_train.shape[0], -1))
val_pred = best_model.predict(X_val.reshape(X_val.shape[0], -1))
test_pred = best_model.predict(X_test.reshape(X_test.shape[0], -1))

# Inverse scaling
train_pred_inv = scaler.inverse_transform(train_pred.reshape(-1, 1))
y_train_inv = scaler.inverse_transform(y_train.reshape(-1, 1))

val_pred_inv = scaler.inverse_transform(val_pred.reshape(-1, 1))
y_val_inv = scaler.inverse_transform(y_val.reshape(-1, 1))

test_pred_inv = scaler.inverse_transform(test_pred.reshape(-1, 1))
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))


In [ ]:
# Evaluate the model
mae_train = mean_absolute_error(y_train_inv, train_pred_inv)
mae_test = mean_absolute_error(y_test_inv, test_pred_inv)
print("Mean Absolute Error on Training Data:", mae_train)
print("Mean Absolute Error on Testing Data:", mae_test)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100 

# Subset of your data
y_true = y_test_inv
y_pred = test_pred_inv
x = np.arange(len(y_true))

window = 100  # how many points to show at once
lag = 1       # number of steps prediction lags behind

fig, ax = plt.subplots(figsize=(10,6))
line_true, = ax.plot([], [], label="True Values", color="blue", alpha=0.7)
line_pred, = ax.plot([], [], label="Predictions", color="red", alpha=0.7)
ax.set_ylim(min(y_true.min(), y_pred.min())*0.95, max(y_true.max(), y_pred.max())*1.05)
ax.set_xlabel("Index")
ax.set_ylabel("Load")
ax.set_title("Predictions vs True Values (Trailing Window)")
ax.legend()

def update(frame):
    start = max(0, frame - window)
    end = frame
    line_true.set_data(x[start:end], y_true[start:end])
    
    # Prediction lags behind true values
    pred_start = max(0, frame - window - lag)
    pred_end = max(0, frame - lag)
    line_pred.set_data(x[pred_start:pred_end], y_pred[pred_start:pred_end])
    
    ax.set_xlim(x[start], x[end-1] if end > start else x[start]+1)
    return line_true, line_pred

ani = FuncAnimation(
    fig, update,
    frames=range(0, len(x), 10),   # every 10th frame
    interval=20, blit=True
)
HTML(ani.to_jshtml())



In [ ]:
mape = np.mean(np.abs((y_test_inv - test_pred_inv) / y_test_inv)) * 100
print(f"Testing MAPE: {mape:.2f}%")

eps = 1e-6
train_mape = np.mean(np.abs((y_train_inv- train_pred_inv) / (y_train_inv + eps))) * 100
print(f"Traingin MAPE: {mape:.2f}%")

mape = np.mean(np.abs((y_val_inv - val_pred_inv) / y_val_inv)) * 100
print(f"Val MAPE: {mape:.2f}%")



In [ ]:
from sklearn.metrics import r2_score

rmse = np.sqrt(mean_squared_error(y_test_inv, test_pred_inv))
print(f"Testing RMSE: {rmse}")

r2 = r2_score(y_test_inv, test_pred_inv)
print(f"Testing R2: {r2}")

In [ ]:
# import joblib

# joblib.dump(best_model, "svr_model.joblib")


In [ ]:
subset_start = 0
subset_end = 1000
plt.figure(figsize=(12,6))
plt.plot(y_test_inv[subset_start: subset_end], label="True Load", color="black", alpha=0.7)
plt.plot(test_pred_inv[subset_start: subset_end], label="Predictions", color="orange", alpha=0.7)
plt.xlabel("Time Index")
plt.ylabel("Load (MW)")
plt.title("SVR Predictions vs True Load")
plt.legend()
plt.show()

In [ ]:
# from datetime import datetime
# import matplotlib.pyplot as plt
# import meteostat
# from meteostat import Point, Daily, Hourly

# start = datetime(2018, 1, 1, 0, 0)
# end = datetime(2018, 1, 1, 12, 0)

# # Create Point for Vancouver, BC
# vancouver = Point(42.65, -73.75)

# # Get daily data for 2018
# data = Hourly(vancouver, start, end)
# data = data.fetch()

# # Plot line chart including average, minimum and maximum temperature
# data